In [4]:
import os
print(os.listdir('/shared'))

['Untitled.ipynb', 'problem1-reg_data.csv', 'ads_data.csv.zip', 'homeworks', 'lesson_3_data.csv', 'sinAB_PRO', 'problem1-auth_data.csv', 'conspect', 'lesson_2_data.csv', 'image_2022-04-21_13-38-49.png', '.DS_Store', 'problem2.csv', 'invoices.csv', 'order_leads.csv', 'dag_osilyutina_test.py', 'sales_team.csv', 'analytics_interview', 'karpovcourse', 'Res_Tree']


In [5]:
import pandas as pd

# Загрузила только первые 10 000 строк из каждого файла (от загрузки всей БД падало ядро)
reg_sample = pd.read_csv('/shared/problem1-reg_data.csv', sep=';', nrows=10000)
auth_sample = pd.read_csv('/shared/problem1-auth_data.csv', sep=';', nrows=100000)

In [7]:
reg_sample

,reg_ts,uid
0,911382223,1
1,932683089,2
2,947802447,3
3,959523541,4
4,969103313,5
...,...,...
9995,1358884095,11082
9996,1358889351,11084
9997,1358894606,11087
9998,1358899861,11088


In [8]:
auth_sample

,auth_ts,uid
0,911382223,1
1,932683089,2
2,932921206,2
3,933393015,2
4,933875379,2
...,...,...
99995,1366527369,11786
99996,1366527400,7832
99997,1366527922,12140
99998,1366528679,11529


In [15]:
# Оставила только тех пользователей, которые есть в обеих таблицах
common_uids = set(reg_sample['uid']).intersection(set(auth_sample['uid']))

reg_sample = reg_sample[reg_sample['uid'].isin(common_uids)]
auth_sample = auth_sample[auth_sample['uid'].isin(common_uids)]

In [16]:
df = pd.merge(reg_sample, auth_sample, on='uid')

In [17]:
df

,reg_ts,uid,auth_ts
0,911382223,1,911382223
1,932683089,2,932683089
2,932683089,2,932921206
3,932683089,2,933393015
4,932683089,2,933875379
...,...,...,...
96384,1358899861,11088,1365567522
96385,1358899861,11088,1365909078
96386,1358899861,11088,1366086913
96387,1358899861,11088,1366403992


In [18]:
df['reg_sample'] = pd.to_datetime(df['reg_ts'], unit='s').dt.date

In [19]:
df['auth_sample'] = pd.to_datetime(df['auth_ts'], unit='s').dt.date

In [20]:
# Нашла разницу между датами и переводим её в количество дней
df['lifetime'] = (df['auth_sample'] - df['reg_sample']).dt.days

In [23]:
# Сгруппировала данные и посчитала уникальных пользователей в каждой группе
cohort_data = df.groupby(['reg_sample', 'lifetime'], as_index=False) \
                       .agg({'uid': 'nunique'}) \
                       .rename(columns={'uid': 'active_users'})

print(cohort_data.head())

   reg_sample  lifetime  active_users
0  1998-11-18         0             1
1  1999-07-22         0             1
2  1999-07-22         3             1
3  1999-07-22         9             1
4  1999-07-22        14             1


In [25]:
# Построила сводную таблицу (когортную матрицу)
cohort_pivot = cohort_data.pivot(index='reg_sample', columns='lifetime', values='active_users')

# Заменила пропуски (дни, когда никто не зашел) на нули
cohort_pivot = cohort_pivot.fillna(0)

# Посмотрим на получившуюся матрицу
cohort_pivot.head()

lifetime,0,1,2,3,4,5,6,7,8,9,...,4982,4984,4989,4995,5001,5005,5010,5016,5018,5022
reg_sample,,,,,,,,,,,,,,,,,,,,,
1998-11-18,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1999-07-22,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2000-01-13,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2000-05-28,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2000-09-16,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [26]:
# Поделила каждый столбец на столбец с индексом 0 (размер когорты)
retention = cohort_pivot.divide(cohort_pivot[0], axis=0)

# Округлила значения для наглядности (например, до 4 знаков после запятой)
retention = retention.round(4)

# Посмотрим на первые строки получившейся матрицы retention
retention.head()

lifetime,0,1,2,3,4,5,6,7,8,9,...,4982,4984,4989,4995,5001,5005,5010,5016,5018,5022
reg_sample,,,,,,,,,,,,,,,,,,,,,
1998-11-18,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1999-07-22,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2000-01-13,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2000-05-28,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2000-09-16,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [45]:
# Собранная функция
import pandas as pd

def calculate_retention(reg_path, auth_path, start_date=None, end_date=None):
    # 1. Загрузка данных
    reg_sample = pd.read_csv(reg_path, sep=';')
    auth_sample = pd.read_csv(auth_path, sep=';')
    
    # Преобразуем исходные timestamp в даты (normalize сохраняет тип datetime64)
    reg_sample['reg_date'] = pd.to_datetime(reg_sample['reg_ts'], unit='s').dt.normalize()
    auth_sample['auth_date'] = pd.to_datetime(auth_sample['auth_ts'], unit='s').dt.normalize()
    
    # 2. Фильтрация данных по датам для экономии памяти
    if start_date and end_date:
        start_date = pd.to_datetime(start_date)
        end_date = pd.to_datetime(end_date)
        
        # Фильтруем регистрации по выбранному периоду
        reg_sample = reg_sample[(reg_sample['reg_date'] >= start_date) & (reg_sample['reg_date'] <= end_date)]
        # Оставляем только авторизации отфильтрованных пользователей
        auth_sample = auth_sample[auth_sample['uid'].isin(reg_sample['uid'])]
        
    # 3. Объединение таблиц
    df = pd.merge(reg_sample, auth_sample, on='uid')
    
    # 4. Расчет "дня жизни" (lifetime)
    df['lifetime'] = (df['auth_date'] - df['reg_date']).dt.days
    
    # 5. Группировка
    cohort_data = df.groupby(['reg_date', 'lifetime'], as_index=False) \
                           .agg({'uid': 'nunique'}) \
                           .rename(columns={'uid': 'active_users'})
    
    # 6. Сводная таблица (pivot)
    cohort_pivot = cohort_data.pivot(index='reg_date', columns='lifetime', values='active_users')
    cohort_pivot = cohort_pivot.fillna(0)
    
    # 7. Расчет retention в долях
    retention = cohort_pivot.divide(cohort_pivot[0], axis=0)
    retention = retention.round(4)
    
    return retention

In [46]:
 # Пример, проверка работоспособности функции
retention_matrix = calculate_retention(
    reg_path='/shared/problem1-reg_data.csv', 
    auth_path='/shared/problem1-auth_data.csv',
    start_date='2020-09-01',
    end_date='2020-09-07'
)

retention_matrix

lifetime,0,1,2,3,4,5,6,7,8,9,...,13,14,15,16,17,18,19,20,21,22
reg_date,,,,,,,,,,,,,,,,,,,,,
2020-09-01,1.0,0.0202,0.0410,0.0422,0.0460,0.0643,0.0561,0.0643,0.0410,0.0542,...,0.0549,0.0429,0.0441,0.0435,0.0504,0.0416,0.0435,0.0359,0.0435,0.0158
2020-09-02,1.0,0.0252,0.0390,0.0466,0.0567,0.0623,0.0712,0.0630,0.0485,0.0409,...,0.0435,0.0497,0.0529,0.0416,0.0485,0.0390,0.0372,0.0466,0.0264,0.0000
2020-09-03,1.0,0.0233,0.0541,0.0471,0.0654,0.0698,0.0855,0.0635,0.0547,0.0629,...,0.0597,0.0547,0.0534,0.0440,0.0484,0.0465,0.0547,0.0239,0.0000,0.0000
2020-09-04,1.0,0.0201,0.0364,0.0364,0.0515,0.0640,0.0728,0.0496,0.0534,0.0433,...,0.0559,0.0439,0.0452,0.0452,0.0427,0.0477,0.0264,0.0000,0.0000,0.0000
2020-09-05,1.0,0.0276,0.0395,0.0464,0.0545,0.0589,0.0746,0.0489,0.0508,0.0545,...,0.0539,0.0451,0.0514,0.0445,0.0476,0.0251,0.0000,0.0000,0.0000,0.0000
2020-09-06,1.0,0.0313,0.0432,0.0544,0.0563,0.0588,0.0682,0.0607,0.0582,0.0550,...,0.0519,0.0550,0.0457,0.0482,0.0256,0.0000,0.0000,0.0000,0.0000,0.0000
2020-09-07,1.0,0.0294,0.0425,0.0537,0.0506,0.0637,0.0731,0.0525,0.0412,0.0562,...,0.0506,0.0425,0.0506,0.0337,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
